# Analyze 3-Arms Predictions - Complete Markets Only

This notebook:
1. Imports predictions_combined_final.csv
2. Filters to markets that have all three arms (baseline, volume, full_technical)
3. Organizes data with one row per market showing:
   - Market ticker
   - Title
   - Prior (mid_yes)
   - Baseline prediction
   - Volume prediction
   - Full technical prediction

In [1]:
import pandas as pd
import numpy as np

# Load the predictions
df = pd.read_csv('data/output/predictions_combined_final.csv')

print(f"Total predictions: {len(df):,}")
print(f"\nArm distribution:")
print(df['arm'].value_counts())
print(f"\nUnique markets: {df['market_ticker'].nunique():,}")

Total predictions: 5,028

Arm distribution:
arm
baseline          1676
volume            1676
full_technical    1676
Name: count, dtype: int64

Unique markets: 1,676


In [2]:
# Check which markets have all three arms
markets_by_arm = df.groupby('market_ticker')['arm'].apply(set).reset_index()
required_arms = {'baseline', 'volume', 'full_technical'}

# Filter to markets with all three arms
complete_markets = markets_by_arm[markets_by_arm['arm'].apply(lambda x: x == required_arms)]['market_ticker']

print(f"Markets with all three arms: {len(complete_markets):,}")
print(f"Markets missing arms: {df['market_ticker'].nunique() - len(complete_markets):,}")

Markets with all three arms: 1,676
Markets missing arms: 0


In [3]:
# Filter to complete markets only
df_complete = df[df['market_ticker'].isin(complete_markets)].copy()

print(f"Predictions for complete markets: {len(df_complete):,}")
print(f"Expected: {len(complete_markets) * 3:,}")
print(f"Match: {len(df_complete) == len(complete_markets) * 3}")

Predictions for complete markets: 5,028
Expected: 5,028
Match: True


In [4]:
# Pivot to one row per market with all predictions
pivot = df_complete.pivot_table(
    index=['market_ticker', 'title', 'mid_yes'],
    columns='arm',
    values='p_yes',
    aggfunc='first'
).reset_index()

# Rename columns for clarity
pivot.columns.name = None
pivot = pivot.rename(columns={
    'mid_yes': 'prior',
    'baseline': 'p_baseline',
    'volume': 'p_volume',
    'full_technical': 'p_technical'
})

# Reorder columns
pivot = pivot[[
    'market_ticker',
    'title',
    'prior',
    'p_baseline',
    'p_volume',
    'p_technical'
]]

print(f"Final dataset shape: {pivot.shape}")
print(f"Columns: {list(pivot.columns)}")
pivot.head(10)

Final dataset shape: (1676, 6)
Columns: ['market_ticker', 'title', 'prior', 'p_baseline', 'p_volume', 'p_technical']


,market_ticker,title,prior,p_baseline,p_volume,p_technical
0,KXALBUMSALES-WUT-10000,How many album sales will Charlie xcx's 'Wuthe...,0.775,0.70,0.70,0.80
1,KXALBUMSALES-WUT-20000,How many album sales will Charlie xcx's 'Wuthe...,0.605,0.60,0.50,0.50
2,KXALBUMSALES-WUT-25000,How many album sales will Charlie xcx's 'Wuthe...,0.570,0.50,0.50,0.54
3,KXALBUMSALES-WUT-30000,How many album sales will Charlie xcx's 'Wuthe...,0.400,0.40,0.35,0.35
4,KXALBUMSALES-WUT-40000,How many album sales will Charlie xcx's 'Wuthe...,0.260,0.26,0.20,0.30
5,KXALBUMSALES-WUT-45000,How many album sales will Charlie xcx's 'Wuthe...,0.185,0.18,0.15,0.12
6,KXALBUMSALES-WUT-5000,How many album sales will Charlie xcx's 'Wuthe...,0.890,0.89,0.85,0.87
7,KXALBUMSALES-WUT-50000,How many album sales will Charlie xcx's 'Wuthe...,0.155,0.10,0.10,0.16
8,KXALBUMSALES-WUT-55000,How many album sales will Charlie xcx's 'Wuthe...,0.100,0.10,0.05,0.10
9,KXALEAGUEGAME-26FEB06MACPER-MAC,Macarthur vs Perth,0.475,0.47,0.48,0.48


In [5]:
# Summary statistics
print("Summary Statistics:")
print("="*70)
print(pivot[['prior', 'p_baseline', 'p_volume', 'p_technical']].describe())

Summary Statistics:
             prior   p_baseline     p_volume  p_technical
count  1676.000000  1676.000000  1676.000000  1676.000000
mean      0.404415     0.402924     0.402607     0.403553
std       0.295786     0.294886     0.286997     0.292246
min       0.010000     0.010000     0.005000     0.010000
25%       0.155000     0.150000     0.150000     0.150000
50%       0.360000     0.360000     0.360000     0.360000
75%       0.605000     0.600000     0.590000     0.610000
max       0.995000     0.990000     0.990000     0.990000


In [6]:
# Calculate prediction deltas (change from prior)
pivot['delta_baseline'] = pivot['p_baseline'] - pivot['prior']
pivot['delta_volume'] = pivot['p_volume'] - pivot['prior']
pivot['delta_technical'] = pivot['p_technical'] - pivot['prior']

print("Average changes from prior:")
print(f"Baseline: {pivot['delta_baseline'].mean():.6f}")
print(f"Volume: {pivot['delta_volume'].mean():.6f}")
print(f"Technical: {pivot['delta_technical'].mean():.6f}")
print()
print("Markets where each arm moved most:")
print(f"Baseline |delta| > 0.1: {(abs(pivot['delta_baseline']) > 0.1).sum():,}")
print(f"Volume |delta| > 0.1: {(abs(pivot['delta_volume']) > 0.1).sum():,}")
print(f"Technical |delta| > 0.1: {(abs(pivot['delta_technical']) > 0.1).sum():,}")

Average changes from prior:
Baseline: -0.001492
Volume: -0.001808
Technical: -0.000862

Markets where each arm moved most:
Baseline |delta| > 0.1: 0
Volume |delta| > 0.1: 2
Technical |delta| > 0.1: 10


In [7]:
# Save to CSV
output_file = 'data/output/markets_3arms_complete.csv'
pivot.to_csv(output_file, index=False)
print(f"Saved to: {output_file}")
print(f"Shape: {pivot.shape}")
print(f"\nFile ready for analysis!")

Saved to: data/output/markets_3arms_complete.csv
Shape: (1676, 9)

File ready for analysis!


In [8]:
# Quick preview of markets with biggest changes
pivot['max_delta'] = pivot[['delta_baseline', 'delta_volume', 'delta_technical']].abs().max(axis=1)
top_movers = pivot.nlargest(10, 'max_delta')[[
    'title',
    'prior',
    'p_baseline',
    'p_volume',
    'p_technical',
    'max_delta'
]]

print("Top 10 markets with biggest prediction changes:")
print("="*70)
top_movers

Top 10 markets with biggest prediction changes:


,title,prior,p_baseline,p_volume,p_technical,max_delta
1585,New South Wales Blues vs South Australia Redbacks,0.505,0.51,0.52,0.20,0.305
1266,FLA Panthers at TB Lightning: First Goal,0.440,0.44,0.44,0.30,0.140
332,Pro Basketball All Stars?,0.510,0.51,0.48,0.40,0.110
1017,The Citadel at Samford: Total Points,0.405,0.41,0.41,0.30,0.105
1,How many album sales will Charlie xcx's 'Wuthe...,0.605,0.60,0.50,0.50,0.105
1032,Denver at North Dakota St.: Total Points,0.505,0.50,0.53,0.40,0.105
204,LCK Cup 2026: OKSavingsBank BRION vs. DRX Map 2,0.550,0.55,0.45,0.54,0.100
908,Morehead St. at Southeast Missouri St.: Spread,0.400,0.40,0.40,0.30,0.100
972,Southern Indiana at Tennessee-Martin: Spread,0.400,0.40,0.42,0.30,0.100
1040,Denver at North Dakota St.: Total Points,0.400,0.40,0.35,0.30,0.100
